In [ ]:
from google.colab import userdata

In [ ]:
import requests
import time
import datetime
import asyncio
import httpx
import os
from typing import Callable, Dict, List, Optional
import aiohttp
import random

In [ ]:
# Terminal Colors
RED = "\033[31;1m"
GREEN = "\033[32;1m"
YELLOW = "\033[33;1m"
PURPLE = "\033[35;1m"
CYAN = "\033[36;1m"
WHITE = "\033[37;1m"
ORANGE = "\033[38;5;208m"
TEAL = "\033[38;5;37m"
RESET = "\033[0;0m"

In [ ]:
#Synchronous
def fetch_sync(url):
  print(f"{PURPLE}Starting the request to {url}")
  response = requests.get(url)
  print(f"{ORANGE}Finished request to {url}")
  return response

In [ ]:
url = "https://jsonplaceholder.typicode.com/posts"

In [ ]:
start_time = time.perf_counter()

for i in range(1,101):
  fetch_sync(f"{url}/{i}")

end_time = time.perf_counter()

Starting the request to https://jsonplaceholder.typicode.com/posts/1
Finished request to https://jsonplaceholder.typicode.com/posts/1
Starting the request to https://jsonplaceholder.typicode.com/posts/2
Finished request to https://jsonplaceholder.typicode.com/posts/2
Starting the request to https://jsonplaceholder.typicode.com/posts/3
Finished request to https://jsonplaceholder.typicode.com/posts/3
Starting the request to https://jsonplaceholder.typicode.com/posts/4
Finished request to https://jsonplaceholder.typicode.com/posts/4
Starting the request to https://jsonplaceholder.typicode.com/posts/5
Finished request to https://jsonplaceholder.typicode.com/posts/5
Starting the request to https://jsonplaceholder.typicode.com/posts/6
Finished request to https://jsonplaceholder.typicode.com/posts/6
Starting the request to https://jsonplaceholder.typicode.com/posts/7
Finished request to https://jsonplaceholder.typicode.com/posts/7
Starting the request to https://jsonplaceholder.typicode.com/p

In [ ]:
print(f"{GREEN}Total time: {end_time - start_time:.2f} seconds")

Total time: 18.96 seconds


In [ ]:
#Asynchronous
async def fetch_async(client, url):
  print(f"{PURPLE}Starting the request to {url}")
  response = await client.get(url)
  print(f"{ORANGE}Finish request to {url}")
  return response.text

In [ ]:
async def main_async():
  semaphore = asyncio.Semaphore(5)

  tasks_to_create = 100

  async with httpx.AsyncClient() as client:
    async def fetch_with_semaphore(current_url):
      async with semaphore:
        return await fetch_async(client, current_url)

      #Creating 100 tasks, each calling the url with semaphore protection
      tasks = [fetch_with_semaphore(url) for _ in range(0, tasks_to_create)]
      results = await asyncio.gather(*tasks)

In [ ]:
start_time = time.perf_counter()

await main_async()

end_time = time.perf_counter()

In [ ]:
print(f"{GREEN}Total time: {end_time - start_time:.2f} seconds")

Total time: 0.58 seconds


In [ ]:
async def greet(name:str, delay:int)->str:
  print(f"{GREEN}Hello {name}")
  await asyncio.sleep(delay)
  print(f"{ORANGE} Bye {name}")
  return f"Processed {name}"

async def main():
  #Single co-routine
  result = await greet("Adam", 2)
  #Multiple co-routines
  results = await asyncio.gather(greet("Alex", 5), greet("Brian", 3), greet("Tom", 2))
  print(f"All greetings completed: {results}")


In [ ]:
await main()

Hello Adam
 Bye Adam
Hello Alex
Hello Brian
Hello Tom
 Bye Tom
 Bye Brian
 Bye Alex
All greetings completed: ['Processed Alex', 'Processed Brian', 'Processed Tom']


In [ ]:
#Event Loop
async def monitor_task(name:str, duration:int)->str:
  print(f"{GREEN}{name} starting...")
  await asyncio.sleep(duration)
  print(f"{ORANGE}[{name}] completed after {duration} seconds")
  return f"{WHITE}{name}_result"

async def main():
  #Creating task manually
  task1 = asyncio.create_task(monitor_task("Task-1", 3))
  task2 = asyncio.create_task(monitor_task("Task-2", 2))

  #Tasks should follow immediately after creation
  result1 = await task1
  await asyncio.sleep(1)
  result2 = await task2

  print(f"{TEAL}Results: {result1}, {result2}")

In [ ]:
await main()

Task-1 starting...
Task-2 starting...
[Task-2] completed after 2 seconds
[Task-1] completed after 3 seconds
Results: Task-1_result, Task-2_result


In [ ]:
weather_api_key = userdata.get("WEATHER_API_KEY")
city = "New York"
units = "imperial"

In [ ]:
url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={weather_api_key}&units={units}&lang=en"

In [ ]:
#Limiting Concurrent Connections
async def fetch_with_limit(sem, session, url, idx):
  print(f"{TEAL}[Task {idx}] Waiting for permit...")
  async with sem:
    print(f"{GREEN}[Task {idx} Got Permit]")
    response = await session.get(url)
    print(f"{RED}[Task {idx} Release Permit]")
    return response.json()

async def main():
  #Allowing 5 max concurrent requests
  semaphore = asyncio.Semaphore(5)

  urls = [url for i in range(0, 60)]

  async with httpx.AsyncClient() as session:
    tasks = [fetch_with_limit(semaphore, session, url, i) for i, url in enumerate(urls, 1)]
    start = time.perf_counter()
    results = await asyncio.gather(*tasks)
    end = time.perf_counter()
  print(f"{WHITE}Fetched {len(results)} item in {end-start:.2f} seconds")

In [ ]:
await main()

[Task 1] Waiting for permit...
[Task 1 Got Permit]
[Task 2] Waiting for permit...
[Task 2 Got Permit]
[Task 3] Waiting for permit...
[Task 3 Got Permit]
[Task 4] Waiting for permit...
[Task 4 Got Permit]
[Task 5] Waiting for permit...
[Task 5 Got Permit]
[Task 6] Waiting for permit...
[Task 7] Waiting for permit...
[Task 8] Waiting for permit...
[Task 9] Waiting for permit...
[Task 10] Waiting for permit...
[Task 11] Waiting for permit...
[Task 12] Waiting for permit...
[Task 13] Waiting for permit...
[Task 14] Waiting for permit...
[Task 15] Waiting for permit...
[Task 16] Waiting for permit...
[Task 17] Waiting for permit...
[Task 18] Waiting for permit...
[Task 19] Waiting for permit...
[Task 20] Waiting for permit...
[Task 21] Waiting for permit...
[Task 22] Waiting for permit...
[Task 23] Waiting for permit...
[Task 24] Waiting for permit...
[Task 25] Waiting for permit...
[Task 26] Waiting for permit...
[Task 27] Waiting for permit...
[Task 28] Waiting for permit...
[Task 29] Wai

In [ ]:
#Running multiple async tasks
async def task_1():
  print(f"{TEAL}Task 1 is starting")
  await asyncio.sleep(5)
  print(f"{ORANGE}Task 1 finished")

async def task_2():
  print(f"{TEAL}Task 2 is starting")
  await asyncio.sleep(3)
  print(f"{ORANGE}Task 2 finished")

async def task_3():
  print(f"{TEAL}Task 3 is starting")
  await asyncio.sleep(2)
  print(f"{ORANGE}Task 3 finished")

In [ ]:
async def async_tasks_main():
  await asyncio.gather(task_1(), task_2(), task_3())

In [ ]:
await async_tasks_main()

Task 1 is starting
Task 2 is starting
Task 3 is starting
Task 3 finished
Task 2 finished
Task 1 finished


In [ ]:
user_url = "https://jsonplaceholder.typicode.com/users/"

In [ ]:
#Managing multiple async tasks
async def fetch_user(user_id:int)->None:
  print(f"{GREEN}Fetching data for user {user_id}")
  await asyncio.sleep(2) #Simulating network delay
  print(f"{YELLOW}Data for user {user_id} retrieved")

In [ ]:
async def main_fetch_user():
  task1 = asyncio.create_task(fetch_user(1))
  task2 = asyncio.create_task(fetch_user(2))

  await task1
  await task2

In [ ]:
await main_fetch_user()

Fetching data for user 1
Fetching data for user 2
Data for user 1 retrieved
Data for user 2 retrieved


In [ ]:
#Error Handling, Rate-Limiting, and Performance Optimization
async def fetch_httpx(url:str):
  try:
    async with httpx.AsyncClient() as client:
      response = await client.get(url, timeout=5.0)
      response.raise_for_status()
      return response.text
  except httpx.TimeoutException:
    print(f"{RED}Request timed out")
  except httpx.HTTPStatusError as e:
    print(f"{RED}HTTP Error {e.response.status_code}: {e}")
  except httpx.RequestError as e:
    print(f"{RED}Network/request error: {e}")
  except Exception as e:
    print(f"{RED}Unexpected error: {e}")

In [ ]:
async def main_fetch_httpx():
  result = await fetch_httpx(url)
  print(result)

In [ ]:
await fetch_httpx(url)

'{"coord":{"lon":-74.006,"lat":40.7143},"weather":[{"id":800,"main":"Clear","description":"clear sky","icon":"01n"}],"base":"stations","main":{"temp":81.54,"feels_like":81.88,"temp_min":73.94,"temp_max":85.96,"pressure":1017,"humidity":47,"sea_level":1017,"grnd_level":1017},"visibility":10000,"wind":{"speed":12.66,"deg":230},"clouds":{"all":0},"dt":1779063853,"sys":{"type":1,"id":4610,"country":"US","sunrise":1779010635,"sunset":1779062869},"timezone":-14400,"id":5128581,"name":"New York","cod":200}'

In [ ]:
#Rate Limiting
#Retrying with exponential backoff
async def fetch_with_retries(url, retries=3, backoff_factor=1):
  for attempt in range(0, retries):
    try:
      async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
          response.raise_for_status()
          return await response.text()
    except aiohttp.ClientError as e:
      wait_time = backoff_factor * (2 ** attempt) + random.uniform(0, 0.5)
      print(f"{RED}Error: {e}. Retrying in {wait_time:.2f} seconds...")
      await asyncio.sleep(wait_time)
  raise Exception(f"{RED}Failed after {retries} retries")
  return None

In [ ]:
async def fetch_with_retries_main():
  url = "jsonplaceholder.typicode.com/posts/101"
  result = await fetch_with_retries(url)
  print(result)

In [ ]:
await fetch_with_retries_main()

Error: jsonplaceholder.typicode.com/posts/101. Retrying in 1.08 seconds...
Error: jsonplaceholder.typicode.com/posts/101. Retrying in 2.31 seconds...
Error: jsonplaceholder.typicode.com/posts/101. Retrying in 4.04 seconds...


Exception: [31;1mFailed after 3 retries

In [ ]:
#Performance Optimization
#Connection Pooling
async def fetch(session, url):
  async with session.get(url) as response:
    return await response.text()

async def fetch_main():
  urls = [f"https://jsonplaceholder.typicode.com/posts/{i}" for i in range(1, 21)]
  async with aiohttp.ClientSession() as session:
    tasks = [fetch(session, url) for url in urls]
    results = await asyncio.gather(*tasks)
    display(results)

In [ ]:
await fetch_main()

['{\n  "userId": 1,\n  "id": 1,\n  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",\n  "body": "quia et suscipit\\nsuscipit recusandae consequuntur expedita et cum\\nreprehenderit molestiae ut ut quas totam\\nnostrum rerum est autem sunt rem eveniet architecto"\n}',
 '{\n  "userId": 1,\n  "id": 2,\n  "title": "qui est esse",\n  "body": "est rerum tempore vitae\\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\\nqui aperiam non debitis possimus qui neque nisi nulla"\n}',
 '{\n  "userId": 1,\n  "id": 3,\n  "title": "ea molestias quasi exercitationem repellat qui ipsa sit aut",\n  "body": "et iusto sed quo iure\\nvoluptatem occaecati omnis eligendi aut ad\\nvoluptatem doloribus vel accusantium quis pariatur\\nmolestiae porro eius odio et labore et velit aut"\n}',
 '{\n  "userId": 1,\n  "id": 4,\n  "title": "eum et est occaecati",\n  "body": "ullam et saepe reiciendis vo